🌍 TravelGenie AI

Intelligent Multi-Agent Travel Planner

Install Libraries

In [ ]:
!pip install -q openai gradio requests pandas

Import Libraries

In [ ]:
import requests
import pandas as pd
import gradio as gr

from openai import OpenAI
from google.colab import userdata

Load Groq API

In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

Memory

In [ ]:
memory = {}

Planner Agent

This agent understands the user's travel request and creates an execution plan.

In [ ]:
def planner_agent(task):

    prompt = f"""
You are an Expert AI Travel Planner.

Your responsibilities:

1. Understand the user's travel request.
2. Break the task into logical steps.
3. Identify what information is needed.
4. Return ONLY a numbered execution plan.

User Request:

{task}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an Expert AI Travel Planner."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1000
    )

    plan = response.choices[0].message.content.strip()

    memory["task"] = task
    memory["plan"] = plan

    return plan

Test Planner Agent

In [ ]:
task = """
Plan a 5-day family trip.

Destination: Ooty

Starting City: Chennai

Budget: ₹30,000

Travel Month: December
"""

print("=" * 60)
print("PLANNER AGENT")
print("=" * 60)

plan = planner_agent(task)

print(plan)

PLANNER AGENT
1. Determine the number of family members traveling to calculate accommodation and transportation costs.
2. Research and book a suitable mode of transportation from Chennai to Ooty (train, bus, or car) within the given budget.
3. Explore Ooty's top attractions and activities suitable for a 5-day family trip in December, considering weather conditions.
4. Shortlist budget-friendly accommodations in Ooty (hotels, resorts, or homestays) that fit the family's size and budget.
5. Create a daily itinerary for the 5-day trip, including travel time, sightseeing, and leisure activities.
6. Estimate food and miscellaneous expenses for the trip to ensure the total cost stays within ₹30,000.
7. Book accommodations and transportation in advance to avoid peak season rates and availability issues.
8. Research any additional costs, such as entry fees for attractions or activities, and factor them into the overall budget.


Memory Check

After running the test, verify that the planner stored its output:



In [ ]:
print(memory)

{'task': '\nPlan a 5-day family trip.\n\nDestination: Ooty\n\nStarting City: Chennai\n\nBudget: ₹30,000\n\nTravel Month: December\n', 'plan': "1. Determine the number of family members traveling to calculate accommodation and transportation costs.\n2. Research and book a suitable mode of transportation from Chennai to Ooty (train, bus, or car) within the given budget.\n3. Explore Ooty's top attractions and activities suitable for a 5-day family trip in December, considering weather conditions.\n4. Shortlist budget-friendly accommodations in Ooty (hotels, resorts, or homestays) that fit the family's size and budget.\n5. Create a daily itinerary for the 5-day trip, including travel time, sightseeing, and leisure activities.\n6. Estimate food and miscellaneous expenses for the trip to ensure the total cost stays within ₹30,000.\n7. Book accommodations and transportation in advance to avoid peak season rates and availability issues.\n8. Research any additional costs, such as entry fees for

Research Agent

This agent will convert the plan into useful travel information

Destination Research Agent, which will gather information about:

Best tourist attractions
Climate
Local food
Transportation
Shopping
Best time to visit

In [ ]:
def research_agent(task, plan):

    prompt = f"""
You are an Expert Travel Research Agent.

Your responsibilities:

1. Read the travel request.
2. Read the execution plan.
3. Research the destination.

Provide the following:

1. Destination Overview
2. Best Time to Visit
3. Famous Tourist Attractions
4. Local Foods to Try
5. Transportation Options
6. Shopping Places
7. Travel Tips

Return the information in a well-formatted report.

Travel Request:
{task}

Execution Plan:
{plan}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are an Expert Travel Research Agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3,
        max_tokens=2000
    )

    research = response.choices[0].message.content.strip()

    memory["research"] = research

    return research

Test Research Agent

In [ ]:
print("="*60)
print("RESEARCH AGENT")
print("="*60)

research = research_agent(task, plan)

print(research)

RESEARCH AGENT
**Ooty Family Trip Report**

### Destination Overview

Ooty, also known as Udhagamandalam, is a popular hill station in the Nilgiri Hills of Tamil Nadu, India. It is known for its stunning natural beauty, pleasant climate, and rich cultural heritage. Ooty is an ideal destination for a family trip, offering a range of activities and attractions that cater to all ages.

### Best Time to Visit

The best time to visit Ooty is from October to February, with December being a great month to experience the town's festive atmosphere and mild winter climate. The average temperature in December ranges from 8°C to 15°C, making it perfect for outdoor activities and sightseeing.

### Famous Tourist Attractions

1. **Ooty Lake**: A scenic lake with boating facilities and a popular spot for picnics.
2. **Doddabetta Peak**: The highest point in the Nilgiri Hills, offering breathtaking views of the surrounding landscape.
3. **Botanical Gardens**: A beautiful garden with a wide variety of 

Check Memory

In [ ]:
print(memory.keys())

dict_keys(['task', 'plan', 'research'])


Budget Agent

In [ ]:
def budget_agent(task, research):

    prompt = f"""
You are an AI Budget Planner.

Responsibilities:

1. Estimate transportation cost.
2. Estimate hotel cost.
3. Estimate food cost.
4. Estimate sightseeing cost.
5. Estimate shopping cost.
6. Give the total estimated budget.

Return a neat report.

Travel Request:

{task}

Research:

{research}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":"You are an Expert Travel Budget Planner."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.2,
        max_tokens=1200
    )

    budget = response.choices[0].message.content.strip()

    memory["budget"] = budget

    return budget

In [ ]:
print("="*60)
print("BUDGET AGENT")
print("="*60)

budget = budget_agent(task, research)

print(budget)

BUDGET AGENT
**Ooty Family Trip Budget Report**

### Introduction

This report provides a detailed breakdown of the estimated costs for a 5-day family trip to Ooty, starting from Chennai, within a budget of ₹30,000.

### Estimated Costs

1. **Transportation Cost**: ₹8,000
	* This includes the cost of train or bus fare from Chennai to Ooty, as well as any additional transportation costs within Ooty.
2. **Hotel Cost**: ₹10,000
	* This includes the cost of a budget-friendly hotel or homestay for 5 nights, with an average cost of ₹2,000 per night.
3. **Food Cost**: ₹4,000
	* This includes the estimated cost of meals, snacks, and beverages for 5 days, with an average cost of ₹800 per day.
4. **Sightseeing Cost**: ₹2,000
	* This includes the estimated cost of entry fees, boating, and other activities at tourist attractions.
5. **Shopping Cost**: ₹2,000
	* This includes the estimated cost of souvenirs, handicrafts, and other shopping expenses.

### Total Estimated Budget

The total estimated 

Memory

In [ ]:
print(memory.keys())

dict_keys(['task', 'plan', 'research', 'budget'])


🗓️ Itinerary Agent

This agent generates a complete day-wise travel plan.

In [ ]:
def itinerary_agent(task, research, budget):

    prompt = f"""
You are an Expert Travel Itinerary Planner.

Your responsibilities:

1. Read the travel request.
2. Read the destination research.
3. Read the estimated budget.
4. Create a detailed day-wise itinerary.

Include:

• Morning
• Afternoon
• Evening
• Recommended food
• Estimated daily expense
• Travel tips

Travel Request:
{task}

Research:
{research}

Budget:
{budget}

Return a professional itinerary.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":"You are an Expert Travel Planner."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.3,
        max_tokens=2500
    )

    itinerary = response.choices[0].message.content.strip()

    memory["itinerary"] = itinerary

    return itinerary

In [ ]:
print("="*60)
print("ITINERARY AGENT")
print("="*60)

itinerary = itinerary_agent(
    task,
    research,
    budget
)

print(itinerary)

ITINERARY AGENT
**Ooty Family Trip Itinerary**

### Day 1: Chennai to Ooty

* **Morning**: Depart from Chennai by train (Nilgiri Express) or bus to Mettupalayam. From Mettupalayam, take a toy train to Ooty.
* **Afternoon**: Check-in to the hotel and freshen up. Visit the **Ooty Lake** for boating and a picnic.
* **Evening**: Explore the **Charring Cross** shopping area and try some **Nilgiri Tea** at a local tea stall.
* **Recommended Food**: Try some **South Indian Cuisine** at a local restaurant for dinner.
* **Estimated Daily Expense**: ₹4,500 (transportation: ₹2,000, hotel: ₹1,500, food: ₹800, sightseeing: ₹200)
* **Travel Tips**: Book your train or bus tickets in advance to avoid peak season rates. Pack warm clothing for the winter months.

### Day 2: Ooty

* **Morning**: Visit the **Doddabetta Peak** for breathtaking views of the surrounding landscape.
* **Afternoon**: Explore the **Botanical Gardens** and enjoy the tranquil atmosphere.
* **Evening**: Visit the **Wenlock Downs** 

Check Memory

In [ ]:
print(memory.keys())

dict_keys(['task', 'plan', 'research', 'budget', 'itinerary'])


Reviewer Agent

This agent reviews the complete travel plan before presenting it to the user.

In [ ]:
def reviewer_agent(plan, research, budget, itinerary):

    prompt = f"""
You are a Senior Travel Planner.

Review the complete travel plan.

Check:

1. Is the itinerary realistic?
2. Is the budget reasonable?
3. Are important tourist places included?
4. Are travel tips useful?
5. Are there missing recommendations?
6. Suggest improvements if needed.

Return:

Overall Rating (1-10)

Strengths

Weaknesses

Suggestions

Travel Plan

{plan}

Research

{research}

Budget

{budget}

Itinerary

{itinerary}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":"You are an Expert Travel Reviewer."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.2,
        max_tokens=1800
    )

    review = response.choices[0].message.content.strip()

    memory["review"] = review

    return review

Test Reviewer Agent

In [ ]:
print("="*60)
print("REVIEWER AGENT")
print("="*60)

review = reviewer_agent(
    plan,
    research,
    budget,
    itinerary
)

print(review)

REVIEWER AGENT
**Overall Rating: 8/10**

**Strengths:**

1. The travel plan is well-structured and easy to follow.
2. The itinerary includes a good mix of sightseeing, adventure, and relaxation activities.
3. The budget breakdown is detailed and realistic.
4. The travel tips and recommendations are helpful and practical.

**Weaknesses:**

1. The itinerary is a bit packed, with some days having multiple activities that may be tiring for a family trip.
2. The budget breakdown does not include any contingency funds for unexpected expenses.
3. Some activities, such as the cultural show and campfire, may not be suitable for all family members.
4. The itinerary does not include any free time for relaxation or spontaneity.

**Suggestions:**

1. Consider adding some free time to the itinerary to allow for relaxation or spontaneity.
2. Include contingency funds in the budget breakdown to account for unexpected expenses.
3. Provide more options for activities that cater to different interests an

In [ ]:
print(memory.keys())

dict_keys(['task', 'plan', 'research', 'budget', 'itinerary', 'review'])


build the Main AI Controller

Instead of calling every agent manually, we'll create one function:

Master Agent (Orchestrator) that connects everything together.

In [ ]:
def travel_genie(destination, days, budget, travel_type):

    task = f"""
Destination : {destination}

Number of Days : {days}

Budget : ₹{budget}

Travel Type : {travel_type}
"""

    # Planner
    plan = planner_agent(task)

    # Research
    research = research_agent(task, plan)

    # Budget
    budget_report = budget_agent(task, research)

    # Itinerary
    itinerary = itinerary_agent(
        task,
        research,
        budget_report
    )

    # Reviewer
    review = reviewer_agent(
        plan,
        research,
        budget_report,
        itinerary
    )

    final_report = f"""
# 🌍 TravelGenie AI

========================================

## 📋 Execution Plan

{plan}

========================================

## 🔍 Destination Research

{research}

========================================

## 💰 Budget Estimate

{budget_report}

========================================

## 🗓 Day-wise Itinerary

{itinerary}

========================================

## ✅ Reviewer Feedback

{review}

========================================
"""

    return final_report

Test the Entire AI

In [ ]:
result = travel_genie(
    destination="Ooty",
    days=5,
    budget=30000,
    travel_type="Family"
)

print(result)


# 🌍 TravelGenie AI


## 📋 Execution Plan

1. Determine the best time to visit Ooty based on weather and tourist season to plan accordingly.
2. Research and book suitable family-friendly accommodations in Ooty within the given budget of ₹30000 for 5 days.
3. Identify top family-friendly attractions and activities in Ooty, such as Ooty Lake, Botanical Gardens, and Doddabetta Peak.
4. Plan a daily itinerary for the 5-day trip, including travel time between attractions and potential downtime for relaxation.
5. Calculate the estimated cost of food, transportation, and entry fees for each attraction to ensure the trip stays within budget.
6. Research and book transportation to and from Ooty, including flights, trains, or buses, and local transportation options such as taxis or car rentals.
7. Consider booking a guided tour or hiring a local guide to help navigate Ooty and provide insight into the local culture and history.
8. Purchase travel insurance to cover unexpected medical or travel-r

Planner
      ↓
Research
      ↓
Budget
      ↓
Itinerary
      ↓
Reviewer

Build the Gradio Interface

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=travel_genie,

    inputs=[
        gr.Textbox(
            label="📍 Destination",
            placeholder="e.g. Ooty"
        ),

        gr.Number(
            label="📅 Number of Days",
            value=5
        ),

        gr.Number(
            label="💰 Budget (₹)",
            value=30000
        ),

        gr.Dropdown(
            ["Solo", "Family", "Friends", "Couple"],
            value="Family",
            label="👨‍👩‍👧 Travel Type"
        )
    ],

    outputs=gr.Markdown(label="🌍 Travel Plan"),

    title="🌍 TravelGenie AI",

    description="""
### Intelligent Multi-Agent Travel Planner

Powered by:

• Planner Agent, • Research Agent, • Budget Agent, • Itinerary Agent, • Reviewer Agent

"""
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fe6caaba40c38dd7ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
